# Conversion des données IceTag McGill → format pipeline

**Objectif** : Transformer les données IceTag Fall 2019 (1 ligne/minute, 30 vaches)
en bins de 15 minutes au format identique à `data/brut.csv`.

**Source** : `@IceTag_Compiled_Lameness.xlsx` (30 feuilles, 1 par vache)

**Sortie** : `mcgill_brut.csv` + `mcgill_sls_labels.csv`

## 1. Imports et chemins

In [30]:
import datetime
from collections import defaultdict
from pathlib import Path

import openpyxl
import pandas as pd

# --- Chemins ---
ROOT = Path("..")
ICETAG_XLSX = (
    ROOT / "Données completes" / "Données accelerometres" / "Fall 2019"
    / "Icetag" / "IceTags" / "@IceTag_Compiled_Lameness.xlsx"
)
OUTPUT_CSV = ROOT / "mcgill_brut.csv"
LABELS_CSV = ROOT / "mcgill_sls_labels.csv"

# Vérifier que le fichier source existe
assert ICETAG_XLSX.exists(), f"Fichier introuvable : {ICETAG_XLSX}"
print(f"Source : {ICETAG_XLSX.name} ({ICETAG_XLSX.stat().st_size / 1e6:.1f} MB)")

Source : @IceTag_Compiled_Lameness.xlsx (55.4 MB)


## 2. Labels cliniques SLS (scores de boiterie)

Source : `Exercise Study - SLS Scores.xlsx` (Winter 2019).
Score SLS = Edge + Rest + Shiftwt + Uneven (0 à 8).

- **SLS = 0** → vache saine
- **SLS = 1** → boiterie légère
- **SLS ≥ 2** → boiterie confirmée

⚠️ Les scores datent de jan-mars 2019, les IceTag de nov-déc 2019 (8 mois d'écart).

In [33]:
SLS_LABELS = {
    # Cow: (SLS Baseline jan2019, SLS Midway mar2019, statut)
    821:  (2, 2, "boiteuse_persistante"),
    3444: (2, 4, "boiteuse_aggravee"),
    5874: (2, 1, "boiteuse_amelioree"),
    5857: (2, 1, "boiteuse_amelioree"),
    2081: (1, None, "legere"),
    2063: (1, 0, "legere_guerie"),
    2057: (1, 2, "legere_aggravee"),
    3437: (0, 1, "saine_degradee"),
    5875: (0, 1, "saine_degradee"),
    2078: (0, 0, "saine"),
    5879: (0, 0, "saine"),
    8500: (0, 1, "saine_degradee"),
}

# Afficher
print(f"{len(SLS_LABELS)} vaches avec score SLS connu :")
for cow, (b, m, s) in sorted(SLS_LABELS.items(), key=lambda x: x[1][2]):
    print(f"  Vache {cow:<6} Baseline={b}  Midway={m}  → {s}")

12 vaches avec score SLS connu :
  Vache 3444   Baseline=2  Midway=4  → boiteuse_aggravee
  Vache 5874   Baseline=2  Midway=1  → boiteuse_amelioree
  Vache 5857   Baseline=2  Midway=1  → boiteuse_amelioree
  Vache 821    Baseline=2  Midway=2  → boiteuse_persistante
  Vache 2081   Baseline=1  Midway=None  → legere
  Vache 2057   Baseline=1  Midway=2  → legere_aggravee
  Vache 2063   Baseline=1  Midway=0  → legere_guerie
  Vache 2078   Baseline=0  Midway=0  → saine
  Vache 5879   Baseline=0  Midway=0  → saine
  Vache 3437   Baseline=0  Midway=1  → saine_degradee
  Vache 5875   Baseline=0  Midway=1  → saine_degradee
  Vache 8500   Baseline=0  Midway=1  → saine_degradee


## 3. Fonctions utilitaires

In [36]:
BIN_MINUTES = 15

# Colonnes du fichier source
COL_DATE = 4
COL_TIME = 5
COL_MI = 6
COL_STANDING = 7
COL_LYING = 8
COL_STEPS = 9
COL_LB = 10


def time_to_seconds(t) -> int:
    """Convertir un datetime.time en secondes totales."""
    if t is None:
        return 0
    if isinstance(t, datetime.time):
        return t.hour * 3600 + t.minute * 60 + t.second
    if isinstance(t, datetime.timedelta):
        return int(t.total_seconds())
    return 0


def seconds_to_hms(total_seconds: int) -> str:
    """Convertir des secondes en format H:MM:SS (identique à brut.csv)."""
    h = total_seconds // 3600
    m = (total_seconds % 3600) // 60
    s = total_seconds % 60
    return f"{h}:{m:02d}:{s:02d}"


def make_bin_key(dt: datetime.datetime) -> datetime.datetime:
    """Arrondir un datetime au début du bin de 15 min."""
    minute_bin = (dt.minute // BIN_MINUTES) * BIN_MINUTES
    return dt.replace(minute=minute_bin, second=0, microsecond=0)


print("Fonctions chargées.")
print(f"  time_to_seconds(00:01:30) = {time_to_seconds(datetime.time(0, 1, 30))} sec")
print(f"  seconds_to_hms(900) = {seconds_to_hms(900)}")
print(f"  make_bin_key(19:37) = {make_bin_key(datetime.datetime(2019, 11, 11, 19, 37))}")

Fonctions chargées.
  time_to_seconds(00:01:30) = 90 sec
  seconds_to_hms(900) = 0:15:00
  make_bin_key(19:37) = 2019-11-11 19:30:00


## 4. Lecture et conversion

Pour chaque vache (feuille Excel) :
1. Lire les ~47 000 lignes (1 par minute)
2. Regrouper par bins de 15 minutes
3. Agréger : Steps=somme, MI=somme, Lying=somme secondes, Standing=somme secondes, Transitions=somme LB

In [39]:
%%time

print(f"Lecture de {ICETAG_XLSX.name}...")
wb = openpyxl.load_workbook(str(ICETAG_XLSX), read_only=True, data_only=True)
print(f"  {len(wb.sheetnames)} feuilles (vaches) : {wb.sheetnames[:5]}...\n")

all_rows = []
cow_stats = {}

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    cow_id = sheet_name.strip()

    # --- Lire toutes les lignes minute par minute ---
    minutes = []
    for row in ws.iter_rows(min_row=2, values_only=True):
        date_val = row[COL_DATE]
        time_val = row[COL_TIME]
        if date_val is None or time_val is None:
            continue

        # Construire le datetime
        if isinstance(date_val, datetime.datetime):
            d = date_val.date()
        elif isinstance(date_val, datetime.date):
            d = date_val
        else:
            continue

        if isinstance(time_val, datetime.time):
            t = time_val
        elif isinstance(time_val, datetime.datetime):
            t = time_val.time()
        else:
            continue

        minutes.append({
            "dt": datetime.datetime.combine(d, t),
            "mi": int(row[COL_MI] or 0),
            "standing_s": time_to_seconds(row[COL_STANDING]),
            "lying_s": time_to_seconds(row[COL_LYING]),
            "steps": int(row[COL_STEPS] or 0),
            "lb": int(row[COL_LB] or 0),
        })

    minutes.sort(key=lambda x: x["dt"])

    # --- Agréger par bins de 15 min ---
    bins = defaultdict(lambda: {"mi": 0, "standing_s": 0, "lying_s": 0,
                                 "steps": 0, "lb": 0, "count": 0})
    for m in minutes:
        bk = make_bin_key(m["dt"])
        b = bins[bk]
        b["mi"] += m["mi"]
        b["standing_s"] += m["standing_s"]
        b["lying_s"] += m["lying_s"]
        b["steps"] += m["steps"]
        b["lb"] += m["lb"]
        b["count"] += 1

    # --- Convertir en format brut.csv ---
    rows = []
    for bin_start in sorted(bins.keys()):
        b = bins[bin_start]
        if b["count"] < 10:  # ignorer bins trop incomplets
            continue
        bin_end = bin_start + datetime.timedelta(minutes=BIN_MINUTES)
        transitions = b["lb"]
        rows.append({
            "Cow": cow_id,
            "Start": bin_start.strftime("%Y-%m-%d %H:%M:%S"),
            "End": bin_end.strftime("%Y-%m-%d %H:%M:%S"),
            "Steps": b["steps"],
            "Motion Index": b["mi"],
            "Lying Time": seconds_to_hms(b["lying_s"]),
            "Standing Time": seconds_to_hms(b["standing_s"]),
            "Transitions": transitions,
            "Transitions Up": transitions // 2,
            "Transitions Down": transitions - transitions // 2,
        })

    all_rows.extend(rows)

    # Stats
    sls = SLS_LABELS.get(int(cow_id), (None, None, "pas_de_score"))
    dates = set(r["Start"][:10] for r in rows)
    cow_stats[cow_id] = {
        "bins": len(rows), "days": len(dates),
        "first": min(dates) if dates else "",
        "last": max(dates) if dates else "",
        "sls_baseline": sls[0], "sls_midway": sls[1], "statut": sls[2],
    }
    status_icon = {"boiteuse": "🔴", "legere": "🟡", "saine": "🟢", "pas": "⚪"}
    icon = next((v for k, v in status_icon.items() if sls[2].startswith(k)), "⚪")
    print(f"  {icon} Vache {cow_id:<6} → {len(rows):>4} bins, {len(dates)} jours  [{sls[2]}]")

wb.close()
print(f"\n✅ {len(all_rows)} bins totaux pour {len(cow_stats)} vaches")

Lecture de @IceTag_Compiled_Lameness.xlsx...
  30 feuilles (vaches) : ['821', '2041', '2057', '2062', '2063']...

  🔴 Vache 821    → 3145 bins, 34 jours  [boiteuse_persistante]
  ⚪ Vache 2041   → 3145 bins, 34 jours  [pas_de_score]
  🟡 Vache 2057   → 3144 bins, 34 jours  [legere_aggravee]
  ⚪ Vache 2062   → 3144 bins, 34 jours  [pas_de_score]
  🟡 Vache 2063   → 3140 bins, 34 jours  [legere_guerie]
  ⚪ Vache 2066   → 3142 bins, 34 jours  [pas_de_score]
  🟢 Vache 2078   → 3140 bins, 34 jours  [saine]
  🟡 Vache 2081   → 3141 bins, 34 jours  [legere]
  ⚪ Vache 3435   → 3140 bins, 34 jours  [pas_de_score]
  🟢 Vache 3437   → 3145 bins, 34 jours  [saine_degradee]
  🔴 Vache 3444   → 3142 bins, 34 jours  [boiteuse_aggravee]
  ⚪ Vache 5327   → 3145 bins, 34 jours  [pas_de_score]
  ⚪ Vache 5854   → 3145 bins, 34 jours  [pas_de_score]
  🔴 Vache 5857   → 3145 bins, 34 jours  [boiteuse_amelioree]
  ⚪ Vache 5862   → 3146 bins, 34 jours  [pas_de_score]
  ⚪ Vache 5865   → 3144 bins, 34 jours  [pas_de_s

## 5. Vérification des données

In [42]:
df = pd.DataFrame(all_rows)
df = df.sort_values(["Cow", "Start"]).reset_index(drop=True)

print(f"Shape : {df.shape}")
print(f"Colonnes : {list(df.columns)}")
print(f"Période : {df['Start'].min()[:10]} → {df['Start'].max()[:10]}")
print(f"Vaches : {df['Cow'].nunique()}")
print()
df.head(10)

Shape : (93187, 10)
Colonnes : ['Cow', 'Start', 'End', 'Steps', 'Motion Index', 'Lying Time', 'Standing Time', 'Transitions', 'Transitions Up', 'Transitions Down']
Période : 2019-11-11 → 2019-12-14
Vaches : 30



,Cow,Start,End,Steps,Motion Index,Lying Time,Standing Time,Transitions,Transitions Up,Transitions Down
0,2041,2019-11-11 18:45:00,2019-11-11 19:00:00,27,121,0:06:01,0:08:57,7,3,4
1,2041,2019-11-11 19:00:00,2019-11-11 19:15:00,9,18,0:02:05,0:12:55,16,8,8
2,2041,2019-11-11 19:15:00,2019-11-11 19:30:00,16,35,0:01:56,0:13:04,11,5,6
3,2041,2019-11-11 19:30:00,2019-11-11 19:45:00,24,57,0:00:00,0:15:00,0,0,0
4,2041,2019-11-11 19:45:00,2019-11-11 20:00:00,24,34,0:00:05,0:14:55,2,1,1
5,2041,2019-11-11 20:00:00,2019-11-11 20:15:00,16,27,0:01:17,0:13:43,13,6,7
6,2041,2019-11-11 20:15:00,2019-11-11 20:30:00,16,26,0:00:10,0:14:50,2,1,1
7,2041,2019-11-11 20:30:00,2019-11-11 20:45:00,30,48,0:00:50,0:14:10,9,4,5
8,2041,2019-11-11 20:45:00,2019-11-11 21:00:00,16,46,0:05:26,0:09:34,3,1,2
9,2041,2019-11-11 21:00:00,2019-11-11 21:15:00,0,0,0:15:00,0:00:00,0,0,0


In [44]:
# Comparer avec le format brut.csv original
brut_original = pd.read_csv(Path("..") / ".." / "data" / "brut.csv", nrows=5)

print("=== Format brut.csv original ===")
print(brut_original.head(3).to_string(index=False))
print()
print("=== Format mcgill_brut.csv ===")
print(df.head(3).to_string(index=False))
print()

# Vérifier que les colonnes sont identiques
assert list(df.columns) == list(brut_original.columns), "⚠️ Colonnes différentes !"
print("✅ Colonnes identiques")

=== Format brut.csv original ===
 Cow               Start                 End  Steps  Motion Index Lying Time Standing Time  Transitions  Transitions Up  Transitions Down
2056 2023-10-16 12:00:00 2023-10-16 12:15:00      0             0    0:15:00       0:00:00            0               0                 0
2056 2023-10-16 12:15:00 2023-10-16 12:30:00      0             0    0:15:00       0:00:00            0               0                 0
2056 2023-10-16 12:30:00 2023-10-16 12:45:00      0             0    0:15:00       0:00:00            0               0                 0

=== Format mcgill_brut.csv ===
 Cow               Start                 End  Steps  Motion Index Lying Time Standing Time  Transitions  Transitions Up  Transitions Down
2041 2019-11-11 18:45:00 2019-11-11 19:00:00     27           121    0:06:01       0:08:57            7               3                 4
2041 2019-11-11 19:00:00 2019-11-11 19:15:00      9            18    0:02:05       0:12:55           16    

In [46]:
# Stats descriptives par vache
print("Statistiques par vache (moyennes journalières) :")
print(f"{'Vache':<8} {'Bins':<6} {'Jours':<6} {'Steps/j':<10} {'MI/j':<10} {'Trans/j':<10} {'Statut SLS'}")
print("-" * 75)

for cow_id in sorted(cow_stats.keys(), key=int):
    s = cow_stats[cow_id]
    cow_df = df[df["Cow"] == cow_id]
    days = s["days"]
    steps_day = cow_df["Steps"].sum() / max(days, 1)
    mi_day = cow_df["Motion Index"].sum() / max(days, 1)
    trans_day = cow_df["Transitions"].sum() / max(days, 1)
    print(f"{cow_id:<8} {s['bins']:<6} {days:<6} {steps_day:<10.0f} {mi_day:<10.0f} {trans_day:<10.1f} {s['statut']}")

Statistiques par vache (moyennes journalières) :
Vache    Bins   Jours  Steps/j    MI/j       Trans/j    Statut SLS
---------------------------------------------------------------------------
821      3145   34     1298       2513       226.6      boiteuse_persistante
2041     3145   34     681        2659       91.4       pas_de_score
2057     3144   34     579        2257       131.3      legere_aggravee
2062     3144   34     432        2538       361.8      pas_de_score
2063     3140   34     584        1568       203.7      legere_guerie
2066     3142   34     715        2629       18.6       pas_de_score
2078     3140   34     398        960        46.6       saine
2081     3141   34     538        2703       64.1       legere
3435     3140   34     762        1585       76.1       pas_de_score
3437     3145   34     1022       3605       147.2      saine_degradee
3444     3142   34     757        2796       68.6       boiteuse_aggravee
5327     3145   34     96         1065     

## 6. Export

On exporte 2 fichiers :
- `mcgill_brut.csv` → données capteurs au format pipeline
- `mcgill_sls_labels.csv` → labels cliniques par vache

In [49]:
# Export données capteurs
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ {OUTPUT_CSV.name} exporté ({len(df)} lignes, {OUTPUT_CSV.stat().st_size / 1e6:.1f} MB)")

# Export labels SLS
labels_rows = []
for cow_id in sorted(cow_stats.keys(), key=int):
    s = cow_stats[cow_id]
    labels_rows.append({
        "Cow": cow_id,
        "SLS_Baseline_Jan2019": s["sls_baseline"],
        "SLS_Midway_Mar2019": s["sls_midway"],
        "Statut": s["statut"],
        "Bins_15min": s["bins"],
        "Jours": s["days"],
        "Debut": s["first"],
        "Fin": s["last"],
    })

labels_df = pd.DataFrame(labels_rows)
labels_df.to_csv(LABELS_CSV, index=False)
print(f"✅ {LABELS_CSV.name} exporté ({len(labels_df)} vaches)")
print()
labels_df

✅ mcgill_brut.csv exporté (93187 lignes, 6.7 MB)
✅ mcgill_sls_labels.csv exporté (30 vaches)



,Cow,SLS_Baseline_Jan2019,SLS_Midway_Mar2019,Statut,Bins_15min,Jours,Debut,Fin
0,821,2.0,2.0,boiteuse_persistante,3145,34,2019-11-11,2019-12-14
1,2041,NaN,NaN,pas_de_score,3145,34,2019-11-11,2019-12-14
2,2057,1.0,2.0,legere_aggravee,3144,34,2019-11-11,2019-12-14
3,2062,NaN,NaN,pas_de_score,3144,34,2019-11-11,2019-12-14
4,2063,1.0,0.0,legere_guerie,3140,34,2019-11-11,2019-12-14
5,2066,NaN,NaN,pas_de_score,3142,34,2019-11-11,2019-12-14
6,2078,0.0,0.0,saine,3140,34,2019-11-11,2019-12-14
7,2081,1.0,NaN,legere,3141,34,2019-11-11,2019-12-14
8,3435,NaN,NaN,pas_de_score,3140,34,2019-11-11,2019-12-14
9,3437,0.0,1.0,saine_degradee,3145,34,2019-11-11,2019-12-14


## 7. Résumé des groupes cliniques

In [52]:
print("GROUPES CLINIQUES")
print("=" * 60)

groups = labels_df.groupby("Statut")["Cow"].apply(list)
for statut, cows in groups.items():
    icon = {"boiteuse_persistante": "🔴", "boiteuse_aggravee": "🔴",
            "boiteuse_amelioree": "🟠", "legere": "🟡", "legere_aggravee": "🟡",
            "legere_guerie": "🟡", "saine": "🟢", "saine_degradee": "🟢",
            "pas_de_score": "⚪"}.get(statut, "⚪")
    print(f"  {icon} {statut:<25} : {len(cows)} vaches → {cows}")

print(f"\n📁 Fichiers exportés :")
print(f"   {OUTPUT_CSV}")
print(f"   {LABELS_CSV}")
print(f"\n🚀 Prochaine étape : exécuter le pipeline sur mcgill_brut.csv")

GROUPES CLINIQUES
  🔴 boiteuse_aggravee         : 1 vaches → ['3444']
  🟠 boiteuse_amelioree        : 2 vaches → ['5857', '5874']
  🔴 boiteuse_persistante      : 1 vaches → ['821']
  🟡 legere                    : 1 vaches → ['2081']
  🟡 legere_aggravee           : 1 vaches → ['2057']
  🟡 legere_guerie             : 1 vaches → ['2063']
  ⚪ pas_de_score              : 18 vaches → ['2041', '2062', '2066', '3435', '5327', '5854', '5862', '5865', '5870', '5871', '5872', '8501', '8517', '8525', '8526', '8527', '8531', '8536']
  🟢 saine                     : 2 vaches → ['2078', '5879']
  🟢 saine_degradee            : 3 vaches → ['3437', '5875', '8500']

📁 Fichiers exportés :
   ../mcgill_brut.csv
   ../mcgill_sls_labels.csv

🚀 Prochaine étape : exécuter le pipeline sur mcgill_brut.csv
